In [ ]:
%%capture
import os
import pandas as pd
from dj_notebook import activate
from pathlib import Path
env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)

In [ ]:
from PIL import Image
from intecomm_analytics.dataframes import get_df_main_1858
from intecomm_analytics.constants import DM_ALONE, HTN_ALONE, HIV_ALONE, HTN_DM
from intecomm_analytics.utils import (
    get_primary_cohorts_by_categorical_column,
    get_great_table,
    get_columns_for_days_to_event,
)

In [ ]:
df_main = get_df_main_1858(None)

In [ ]:
tbl_dct = get_primary_cohorts_by_categorical_column(df_main, "offstudy_reason")
dftbl = pd.DataFrame(tbl_dct)
mapping = {
    "n": "n",
    "completed_followup": "Completed followup",
    "consent_withdrawal": "Consent withdrawal",
    "dead": "Death",
    "clinical_withdrawal": "Clinical withdrawal",
    "transferred": "Transferred",
    "LTFU": "Lost to followup",
    "pregnant": "Pregnant",
    "OTHER": "Other"
}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=[*mapping.values()], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
# dftbl["variable"] = "country"
# dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfoffstudy_reason = dftbl.copy()

In [ ]:
# days to end of study
df1 = df_main.copy()
dftbl = get_columns_for_days_to_event(df1, "onstudy_days")
dfonstudy= dftbl.copy()

In [ ]:
# death
df1 = df_main.copy()
dftbl = get_columns_for_days_to_event(df1, "death_days_to_event")
dfdeathdays= dftbl.copy()

In [ ]:
# df_main.death_cause.value_counts(dropna=False)
# df_main[df_main.death_cause.notna()][['subject_identifier', 'primary_cohort', 'death_cause']]

In [ ]:
tbl_dct = get_primary_cohorts_by_categorical_column(df_main, "death_cause")
dftbl = pd.DataFrame(tbl_dct)
mapping = {
    "n": "n",
    "completed_followup": "Completed followup",
    "consent_withdrawal": "Consent withdrawal",
    "dead": "Death",
    "clinical_withdrawal": "Clinical withdrawal",
    "transferred": "Transferred",
    "LTFU": "Lost to followup",
    "pregnant": "Pregnant",
    "OTHER": "Other"
}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=[*mapping.values()], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
# dftbl["variable"] = "country"
# dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfdeath_cause = dftbl.copy()

In [ ]:
# viral load measure
df1 = df_main[df_main.primary_cohort.isin([HIV_ALONE])].copy()
dftbl = get_columns_for_days_to_event(df1, "vl_days_to_event", set_zero_to_na=["ncd"])
dfvldays= dftbl.copy()

In [ ]:
# blood pressure measure
# transferred_days_to_event
df1 = df_main[df_main.primary_cohort.isin([HTN_ALONE, HTN_DM])].copy()
dftbl = get_columns_for_days_to_event(df1, "bp_days_to_event", set_zero_to_na=["hiv"])
dfbpdays= dftbl.copy()

In [ ]:
# glucose measure
df1 = df_main[df_main.primary_cohort.isin([DM_ALONE, HTN_DM])].copy()
dftbl = get_columns_for_days_to_event(df1, "glucose_days_to_event", set_zero_to_na=["hiv"])
dfgludays= dftbl.copy()

In [ ]:
# transferred_days_to_event
df1 = df_main.copy()
dftbl = get_columns_for_days_to_event(df1, "transferred_days_to_event")
dftransferreddays= dftbl.copy()

In [ ]:
# ltfu_days_to_event
df1 = df_main.copy()
dftbl = get_columns_for_days_to_event(df1, "ltfu_days_to_event")
dfltfudays= dftbl.copy()

In [ ]:
groupings = [
    # (dfnum, [""]),
    # (dfcountry, ["Site"]),
    (dfoffstudy_reason, ["EoS reasons"]),
    (dfonstudy, ["Days to EoS"]),
    (dfdeathdays, ["Days to death"]),
    (dftransferreddays, ["Days to transfer"]),
    (dfltfudays, ["Days to LTFU"]),
    (dfbpdays, ["Days to endline BP measure"]),
    (dfgludays, ["Days to endline FBG measure"]),
    (dfvldays, ["Days to endline VL measure"]),
]
group_row_headers = [(df, row_headers * (len(df))) for df, row_headers in groupings]
group_row_headers = [row_heading for _, row_headers in group_row_headers for row_heading in row_headers]
# concat all
dftbl_final = pd.concat([df for df, _ in groupings])
dftbl_final = dftbl_final.reset_index(drop=True)
# convert to GT
outcomes_table = get_great_table(dftbl_final, group_row_headers, "Table 1.3: Days to clinical outcomes")
outcomes_table.show()

In [ ]:
# save as png
outcomes_table.save(analysis_folder / "days_to_clinical_outcomes.png")
# export to PDF
image = Image.open(analysis_folder / "days_to_clinical_outcomes.png")
image = image.resize((image.width * 6, image.height * 6), Image.LANCZOS)
image.save(analysis_folder / "days_to_clinical_outcomes.pdf", "PDF", resolution=800, optimize=True, quality=95)